# Pruebas con los algoritmos · borrador

En este notebook pruebo rápido lo que luego paso en limpio a `src/`: cuánto tarda cada
modelo, cuánto inflaban el F1 las dos fugas de la llegada, si `country` y
`booking_changes` podrían ser fugas y si la red de Keras se reinicia de verdad en cada
`fit`.

Es un borrador y no está tan cuidado como los notebooks de `finales/`, pero sus cifras
son las que cito en el README y aquí se pueden volver a sacar. Para no repetir código,
uso `data_loader.preparar()` en lugar de volver a cargar el CSV a mano.

In [1]:
import sys, pathlib

# Subo hasta la raíz del proyecto (la carpeta con src/) para poder importar src.
raiz = next(d for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (d / "src").is_dir())
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

import time

import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline

from src import config, data_loader, model_trainer

pd.set_option("display.max_columns", 40)

d = data_loader.preparar()
X_train, y_train = d["X_train"], d["y_train"]
print(X_train.shape)

      [limpieza] fugas: -4 columnas (reservation_status, reservation_status_date, required_car_parking_spaces, assigned_room_type)
      [limpieza] duplicados exactos: -33.413 filas
      [limpieza] adr negativo: -1 filas
      [limpieza] 0 huéspedes: -165 filas
      [limpieza] total: 119.390 -> 85.811 filas
      train 68.648 / test 17.163 filas | cancelan: 27.64% en train, 27.64% en test
(68648, 27)


## 1. Muestra pequeña para iterar rápido

Con 68.648 filas, 5 folds y 6 modelos, cada prueba tarda demasiado. Para iterar uso una
muestra estratificada de 10.000 filas del train y valido con el resto del train, sin
usar el test.

In [2]:
X_mini, X_resto, y_mini, y_resto = train_test_split(
    X_train, y_train, train_size=10_000, stratify=y_train, random_state=config.SEMILLA,
)
print(X_mini.shape, round(y_mini.mean(), 4))

(10000, 27) 0.2764


## 2. Cuánto tarda cada modelo

Con esto decido qué modelo se queda fuera de la búsqueda de hiperparámetros y qué se
recorta en el modo `--demo`.

In [3]:
tiempos = {}
for nombre in config.MODELOS_ACTIVOS:
    pipeline = Pipeline([("prep", clone(d["preprocesador"])),
                         ("modelo", model_trainer.REGISTRO[nombre]())])
    inicio = time.perf_counter()
    pipeline.fit(X_mini, y_mini)
    tiempos[nombre] = time.perf_counter() - inicio

pd.Series(tiempos).round(1).sort_values().to_frame("segundos (un fit, 10.000 filas)")

,"segundos (un fit, 10.000 filas)"
baseline,0.1
logistica,0.1
arbol,0.1
bosque,0.3
boosting,0.4
red_keras,11.7


La red es con diferencia la más lenta, y además entrena en un solo proceso. Por eso la
dejo fuera de la búsqueda de hiperparámetros, y en `--demo` entrena menos épocas.

## 3. Las fugas de la llegada: cuánto inflan el F1

`required_car_parking_spaces` y `assigned_room_type` se rellenan cuando el cliente llega
(ver `eda_final.ipynb`, sección 2). Para medir cuánto inflan el resultado comparo las
mismas filas del train con y sin ellas, con un `HistGradientBoostingClassifier` de
scikit-learn con sus valores por defecto (salvo la semilla) y los mismos folds que el
pipeline. Es la cifra del apartado 2 del README.

In [4]:
crudo = data_loader.cargar_crudo()

# Las mismas filas que deja limpiar(), pero conservando las columnas de fuga.
filas = crudo.loc[~crudo.drop(columns=config.FUGAS).duplicated()]
huespedes = filas[["adults", "children", "babies"]].fillna(0).sum(axis=1)
filas = filas[(filas["adr"] >= 0) & (huespedes > 0)].reset_index(drop=True)
assert filas.drop(columns=config.FUGAS).equals(data_loader.limpiar(crudo))

X_con_fugas = filas.loc[X_train.index, [*X_train.columns, *config.FUGAS_POSTERIORES]]
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.SEMILLA)

def f1_y_auc(X):
    pipeline = Pipeline([
        ("prep", data_loader.construir_preprocesador(X)),
        ("modelo", HistGradientBoostingClassifier(random_state=config.SEMILLA)),
    ])
    r = cross_validate(pipeline, X, y_train, cv=folds, scoring=["f1", "roc_auc"], n_jobs=-1)
    return round(r["test_f1"].mean(), 3), round(r["test_roc_auc"].mean(), 3)

print("sin las dos (F1, AUC):", f1_y_auc(X_train))
print("con las dos (F1, AUC):", f1_y_auc(X_con_fugas))

      [limpieza] fugas: -4 columnas (reservation_status, reservation_status_date, required_car_parking_spaces, assigned_room_type)
      [limpieza] duplicados exactos: -33.413 filas
      [limpieza] adr negativo: -1 filas
      [limpieza] 0 huéspedes: -165 filas
      [limpieza] total: 119.390 -> 85.811 filas


sin las dos (F1, AUC): (0.678, 0.897)


con las dos (F1, AUC): (0.71, 0.914)


Con ellas el F1 sube de 0,678 a 0,710, pero es una mejora falsa: el modelo no aprende
nada nuevo, porque en el momento de predecir ninguna reserva tiene todavía plaza de
parking anotada ni habitación asignada. Por eso las quito (`config.FUGAS_POSTERIORES`).

## 4. ¿Y country?

`country` es la segunda variable que más pesa en el ganador, detrás de `agent`
(`outputs/feature_importance.png`). Pero tiene algo raro: en los No-Show, el porcentaje
de reservas de Portugal se parece más al de las canceladas que al de las que llegaron,
que es el mismo patrón que tenía el parking.

In [5]:
print("sin country (F1, AUC):", f1_y_auc(X_train.drop(columns="country")))

portugal = filas["country"].eq("PRT")
display(portugal.groupby(filas["reservation_status"]).mean().mul(100).round(1)
        .to_frame("% de reservas de Portugal"))

sin country (F1, AUC): (0.59, 0.839)


,% de reservas de Portugal
reservation_status,
Canceled,39.6
Check-Out,27.6
No-Show,59.4


Sin `country` el F1 baja de 0,678 a 0,590, y la prueba no es tan clara como la del
parking (allí ninguna reserva con plaza estaba cancelada; aquí solo cambia la
proporción). **Decisión:** la dejo, y anoto la duda en el apartado 10 del README: si el
país se corrige en el check-in, parte de su peso sería información que no existe al
reservar.

In [6]:
# Sobre el CSV crudo, como la prueba del parking en eda_final.ipynb.
cambios = crudo["booking_changes"].gt(0)
display(cambios.groupby(crudo["reservation_status"]).mean().mul(100).round(1)
        .to_frame("% con algún cambio"))

print("sin booking_changes (F1, AUC):", f1_y_auc(X_train.drop(columns="booking_changes")))

,% con algún cambio
reservation_status,
Canceled,6.2
Check-Out,20.3
No-Show,14.7


sin booking_changes (F1, AUC): (0.671, 0.893)


Con `booking_changes` la duda es más pequeña. El CSV guarda el número final de cambios,
pero la prueba del parking no la señala: también tienen cambios muchos No-Show
(el 14,7 %), que nunca llegan, así que no se rellena solo al llegar. Y aunque una parte
fuera fuga, pesa poco: sin ella el F1 solo baja de 0,678 a 0,671. **Decisión:** la dejo
(apartado 2 del README).

## 5. La red de Keras: comprobar que se reinicia en cada fit

La red se construye en `fit` y no en `__init__`. `clone()` y `GridSearchCV` crean el
objeto y después le cambian los hiperparámetros con `set_params()`; si la red ya
estuviera construida, ese cambio no le llegaría. Además, el `fit` de Keras no reinicia
los pesos: un segundo `fit` sobre la misma red seguiría entrenándola en lugar de empezar
de cero, y el resultado dependería de lo que se hubiera hecho antes.

Para comprobarlo, entreno la misma red dos veces seguidas: los dos `fit` tienen que dar
exactamente el mismo F1. Al final sigo entrenando la red ya entrenada 5 épocas más, para
ver cuánto cambiaría el F1 si no se reconstruyera.

In [7]:
red = Pipeline([("prep", clone(d["preprocesador"])),
                ("modelo", model_trainer.RedKeras(epocas=5))])
f1s = [f1_score(y_resto, red.fit(X_mini, y_mini).predict(X_resto)) for _ in range(2)]
print("F1 de dos fit seguidos:", f1s, "| iguales:", f1s[0] == f1s[1])

# Lo que pasaría si la red no se reconstruyera: seguir entrenándola 5 épocas más.
keras_red = red.named_steps["modelo"].model_
keras_red.fit(red[:-1].transform(X_mini).astype("float32"), y_mini.to_numpy(dtype="float32"),
              epochs=5, verbose=0)
print("F1 si no se reconstruye:", f1_score(y_resto, red.predict(X_resto)))

F1 de dos fit seguidos: [0.5394872578611546, 0.5394872578611546] | iguales: True


F1 si no se reconstruye: 0.6226454438034337


## Lo que no pruebo aquí

- La **búsqueda de hiperparámetros** ya no está en este notebook: la pasé al pipeline
  (`config.BUSQUEDA = "grid"` y `espacio_busqueda()` de cada modelo).
- El **balanceo de clases** (`class_weight`, SMOTE) lo dejé fuera a propósito: la
  métrica es el F1 con umbral 0,5 y quiero comparar todos los modelos en igualdad de
  condiciones.